# Data pipeline: split and scaling

This notebook builds the data the models will train on. We round the data to 2 decimals, drop the 7 columns that never change, split Dataset 1 into Training, Validation and Calibration blocks by time, scale the continuous sensors, and save the results as files. It comes after the data audit (`Data Audit.ipynb`, `docs/audit.md`) and the exploration (`Explore.ipynb`, `docs/exploration.md`).

- Data used: Dataset 1 (`Training dataset 1.csv`) and Dataset 2 (`Training dataset 2.csv`) only.
- The Test set is not used. It stays untouched until all choices and thresholds are frozen.
- The full write-up of each step is in `docs/pipeline.md`. The notes here are short.

Steps:
0. Setup
0b. Round to 2 decimals and drop the 7 never-changing columns
1. Sort the columns into continuous and on/off
2. Split Dataset 1 by time, with 24-hour gaps
3. Fit the scaler on Training
4. Apply the same scaler to Validation, Calibration and Dataset 2
5. Check that the on/off columns are still 0 and 1
6. Dataset 2 labels (not done yet, see the end of the notebook)
7. Save the files

## 0. Setup

Why: both training files are needed with clean column names and real dates.

What the code does: loads Dataset 1 (`dfT1`) and Dataset 2 (`dfT2`) from `../data/`, strips the spaces from the column names, and turns `DATETIME` into real dates (format day/month/year hour).

In [1]:
import pandas as pd

dfT1 = pd.read_csv("../data/Training dataset 1.csv")
dfT2 = pd.read_csv("../data/Training dataset 2.csv")

dfT1.columns = dfT1.columns.str.strip()
dfT2.columns = dfT2.columns.str.strip()

dfT1["DATETIME"] = pd.to_datetime(dfT1["DATETIME"], format="%d/%m/%y %H")
dfT2["DATETIME"] = pd.to_datetime(dfT2["DATETIME"], format="%d/%m/%y %H")

## 0b. Round to 2 decimals and drop the 7 never-changing columns

Why:
- Rounding: Dataset 1 has up to 9 decimals, but Dataset 2 and the Test set only have 2 (`docs/audit.md`). We round Dataset 1 to 2 decimals too, so Training looks like the data the models will be scored on. Otherwise a model could learn tiny changes that it will never see in Dataset 2 or the Test set.
- Dropping: `S_PU1`, `F_PU3`, `S_PU3`, `F_PU5`, `S_PU5`, `F_PU9` and `S_PU9` never change in Dataset 1 or Dataset 2 (`docs/audit.md`). A column that never changes gives the models nothing to learn from.

We do both before the split and the scaling, so every later step uses the rounded data without these columns.

What the code does: rounds every column except `DATETIME` to 2 decimals in `dfT1` and `dfT2`, drops the 7 columns (`drop_cols`) from both, and prints the new shapes. It also prints how many different values `P_J280` has in Dataset 1 after rounding.

In [2]:
num_cols = dfT1.columns.drop("DATETIME")
dfT1[num_cols] = dfT1[num_cols].round(2)
dfT2[num_cols] = dfT2[num_cols].round(2)

drop_cols = ["S_PU1", "F_PU3", "S_PU3", "F_PU5", "S_PU5", "F_PU9", "S_PU9"]
dfT1 = dfT1.drop(columns=drop_cols)
dfT2 = dfT2.drop(columns=drop_cols)

print(dfT1.shape, dfT2.shape)
print("P_J280 different values in Dataset 1:", dfT1["P_J280"].nunique())

(8761, 38) (4177, 38)
P_J280 different values in Dataset 1: 3


**What this shows:**
- Dataset 1 is now 8,761 rows × 38 columns, and Dataset 2 is 4,177 rows × 38 columns. That is `DATETIME`, 36 features and `ATT_FLAG`.
- After rounding, `P_J280` has only 3 different values in all of Dataset 1.

Caution:
- Pump 3 runs in the Test set, and `S_PU1` is 0 in a few Test rows (`docs/audit.md`). Since we drop these columns, no model will see that.
- `P_J280` moves in a very small range, so rounding leaves only 3 values. See the caution in step 3.

## 1. Continuous and on/off columns

Why: the continuous sensors and the on/off sensors are handled differently. Only the continuous ones get scaled, so we need two separate lists.

What the code does: puts every column that starts with `L_` (tank levels), `F_` (flows) or `P_` (pressures) in `cont_cols`, and every column that starts with `S_` (pump and valve on/off status) in `onoff_cols`. It then prints anything left over (`leftover`), to make sure no sensor is missed.

In [3]:
cont_cols = [c for c in dfT1.columns if c.startswith(("L_", "F_", "P_"))]
onoff_cols = [c for c in dfT1.columns if c.startswith("S_")]

print(len(cont_cols), len(onoff_cols))
print(cont_cols)
print(onoff_cols)

leftover = [c for c in dfT1.columns if c not in cont_cols + onoff_cols]
print(leftover)

28 8
['L_T1', 'L_T2', 'L_T3', 'L_T4', 'L_T5', 'L_T6', 'L_T7', 'F_PU1', 'F_PU2', 'F_PU4', 'F_PU6', 'F_PU7', 'F_PU8', 'F_PU10', 'F_PU11', 'F_V2', 'P_J280', 'P_J269', 'P_J300', 'P_J256', 'P_J289', 'P_J415', 'P_J302', 'P_J306', 'P_J307', 'P_J317', 'P_J14', 'P_J422']
['S_PU2', 'S_PU4', 'S_PU6', 'S_PU7', 'S_PU8', 'S_PU10', 'S_PU11', 'S_V2']
['DATETIME', 'ATT_FLAG']


**What this shows:** 28 continuous columns and 8 on/off columns, so all 36 features are covered. Only `DATETIME` and `ATT_FLAG` are left over, which is what we expect, since they are not sensors.

The 7 dropped columns are no longer in the lists: `F_PU3`, `F_PU5` and `F_PU9` are gone from `cont_cols`, and `S_PU1`, `S_PU3`, `S_PU5` and `S_PU9` are gone from `onoff_cols`.

## 2. Split Dataset 1 by time

Why: we split by time, not at random. Hours next to each other look almost the same, so a random split would put near-copies of Training hours into the later blocks and make the results look better than they are. The GRU and Transformer read 12-hour and 24-hour windows. The 24-hour gaps make sure a window at the start of Validation or Calibration can't reach back into the block before it. The exploration found no drift in Dataset 1 (`docs/exploration.md`), so we keep the 60/20/20 split.

What the code does: cuts Dataset 1 by row position at 60% and 80%. Training (`train`) is the first 60%. Validation (`val`) and Calibration (`cal`) are the next two 20% blocks, but each one starts 24 rows (`gap`) later, so the first 24 hours of each block are dropped. It prints the size and the date range of each block.

In [4]:
n = len(dfT1)
gap = 24

cut1 = int(n * 0.6)
cut2 = int(n * 0.8)

train = dfT1.iloc[:cut1]
val = dfT1.iloc[cut1 + gap:cut2]
cal = dfT1.iloc[cut2 + gap:]

for name, block in [("Training", train), ("Validation", val), ("Calibration", cal)]:
    print(name, len(block), "rows:", block["DATETIME"].min(), "to", block["DATETIME"].max())

print("Total used:", len(train) + len(val) + len(cal), "of", n)

Training 5256 rows: 2014-01-06 00:00:00 to 2014-08-12 23:00:00
Validation 1728 rows: 2014-08-14 00:00:00 to 2014-10-24 23:00:00
Calibration 1729 rows: 2014-10-26 00:00:00 to 2015-01-06 00:00:00
Total used: 8713 of 8761


**What this shows:**
- Training: 5,256 rows, 2014-01-06 00:00 to 2014-08-12 23:00.
- Validation: 1,728 rows, 2014-08-14 00:00 to 2014-10-24 23:00.
- Calibration: 1,729 rows, 2014-10-26 00:00 to 2015-01-06 00:00.

8,713 of the 8,761 rows are used. The rows not used are the two 24-hour gaps (all of 2014-08-13 and all of 2014-10-25). There is no overlap between the blocks.

## 3. Fit the scaler on Training

Why: the sensors use very different units (tank levels, flows, pressures), so we put them all on a 0 to 1 scale. We fit the scaler on Training only. If we used Validation, Calibration or Dataset 2 for this, information from those blocks would leak into the training data.

What the code does: fits a `MinMaxScaler` (`scaler`) on the 28 continuous Training columns. This learns the min and max of each column. It then scales a copy of Training (`train_scaled`) and prints the new min and max of each column (`check`). The on/off columns are not touched.

In [5]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(train[cont_cols])

train_scaled = train.copy()
train_scaled[cont_cols] = scaler.transform(train[cont_cols])

check = train_scaled[cont_cols].agg(["min", "max"]).T
print(check.round(3))

        min  max
L_T1    0.0  1.0
L_T2    0.0  1.0
L_T3    0.0  1.0
L_T4    0.0  1.0
L_T5    0.0  1.0
L_T6    0.0  1.0
L_T7    0.0  1.0
F_PU1   0.0  1.0
F_PU2   0.0  1.0
F_PU4   0.0  1.0
F_PU6   0.0  1.0
F_PU7   0.0  1.0
F_PU8   0.0  1.0
F_PU10  0.0  1.0
F_PU11  0.0  1.0
F_V2    0.0  1.0
P_J280  0.0  1.0
P_J269  0.0  1.0
P_J300  0.0  1.0
P_J256  0.0  1.0
P_J289  0.0  1.0
P_J415  0.0  1.0
P_J302  0.0  1.0
P_J306  0.0  1.0
P_J307  0.0  1.0
P_J317  0.0  1.0
P_J14   0.0  1.0
P_J422  0.0  1.0


**What this shows:** all 28 continuous columns now go from 0 to 1 in Training. Since the never-changing columns were dropped in step 0b, every column has a range to scale.

Caution: `P_J280` has only 3 different values after rounding (step 0b), so in Training its scaled value can only be 0, 0.5 or 1. One 0.01 change in the raw reading is a jump of 0.5 after scaling. So small changes in `P_J280` will look big to the models.

## 4. Apply the same scaler to Validation, Calibration and Dataset 2

Why: all data must be scaled with the same rule, or the models would see different numbers for the same reading.
- Fit = learn the min and max of each column from Training.
- Transform = use that same min and max on other data.

We only transform here. We do not fit again.

What the code does: scales copies of Validation (`val_scaled`), Calibration (`cal_scaled`) and Dataset 2 (`dfT2_scaled`) with the scaler from step 3. It prints the min and max of each continuous column for the three tables (`check2`, `check3`, `check4`). Values below 0 or above 1 mean the reading is outside the Training range.

### 5. On/off columns stay 0 and 1

Why: the 8 `S_` columns are already 0 or 1, so they don't need scaling. We just check that nothing changed them.

What the code does: the last loop in the same cell checks that every `S_` value is 0 or 1 in all four scaled tables, and prints `True` or `False` for each.

In [6]:
val_scaled = val.copy()
val_scaled[cont_cols] = scaler.transform(val[cont_cols])

cal_scaled = cal.copy()
cal_scaled[cont_cols] = scaler.transform(cal[cont_cols])

dfT2_scaled = dfT2.copy()
dfT2_scaled[cont_cols] = scaler.transform(dfT2[cont_cols])

check2 = val_scaled[cont_cols].agg(["min", "max"]).T
print(check2.round(3))

check3 = cal_scaled[cont_cols].agg(["min", "max"]).T
print(check3.round(3))

check4 = dfT2_scaled[cont_cols].agg(["min", "max"]).T
print(check4.round(3))

for name, block in [("Training", train_scaled), ("Validation", val_scaled),
                    ("Calibration", cal_scaled), ("Dataset 2", dfT2_scaled)]:
    print(name, block[onoff_cols].isin([0, 1]).all().all())

          min    max
L_T1    0.020  1.009
L_T2   -0.002  0.996
L_T3    0.008  0.992
L_T4    0.033  0.993
L_T5   -0.003  1.000
L_T6    0.059  1.000
L_T7    0.038  1.000
F_PU1  -0.011  0.995
F_PU2   0.000  0.996
F_PU4   0.000  0.988
F_PU6   0.000  0.000
F_PU7   0.000  1.001
F_PU8   0.000  0.993
F_PU10  0.000  1.004
F_PU11  0.000  0.000
F_V2    0.000  1.019
P_J280  0.000  1.000
P_J269  0.005  1.010
P_J300  0.002  0.986
P_J256  0.007  1.014
P_J289  0.002  0.986
P_J415 -0.015  0.723
P_J302  0.097  1.018
P_J306  0.027  1.008
P_J307  0.103  1.018
P_J317  0.011  0.650
P_J14   0.006  1.020
P_J422  0.002  0.984
          min    max
L_T1    0.033  0.978
L_T2    0.000  1.000
L_T3    0.008  0.996
L_T4    0.019  0.989
L_T5    0.000  0.986
L_T6    0.029  1.000
L_T7    0.061  1.000
F_PU1  -0.003  0.995
F_PU2   0.000  0.996
F_PU4   0.000  1.006
F_PU6   0.000  0.974
F_PU7   0.000  0.998
F_PU8   0.000  1.003
F_PU10  0.000  0.991
F_PU11  0.000  0.000
F_V2    0.000  0.984
P_J280  0.000  1.000
P_J269  0.005

**What this shows:**
- Validation and Calibration stay close to 0 to 1. The values outside are small, for example `F_PU1` −0.011 and `P_J302` 1.018 in Validation, and `P_J306` 1.027 in Calibration.
- Some maxes are well below 1, for example `P_J317` 0.650 in Validation and 0.661 in Calibration. `F_PU6` and `F_PU11` are 0 in Validation, and `F_PU11` is 0 in Calibration, because pumps 6 and 11 don't run in those blocks (`docs/exploration.md`).
- `P_J280` goes from exactly 0 to 1 in both Validation and Calibration, because of its 3 rounded values.
- Dataset 2 goes much further outside the Training range. For example: `F_PU1` down to −1.277, `P_J14` up to 2.860, `P_J302` up to 2.662 and `P_J307` up to 2.642, `L_T1` up to 1.200, `L_T7` down to −0.157, and `P_J280` up to 1.500.
- Step 5: the check prints `True` for Training, Validation, Calibration and Dataset 2, so the on/off columns are still only 0 and 1.

Caution:
- These are only the min and max of each column. We have not checked yet whether these extreme values fall inside the attack windows, so any reason we give for them is only a guess for now.
- `P_J280` up to 1.500 in Dataset 2 is only one 0.01 step above its Training max (see the caution in step 3). Before rounding it went from −0.197 to 1.492, so rounding removed the small values below 0 but the steps are now big.

## 7. Save the files

Why: the training code and the later notebooks should all use the exact same split and scaling. Saving the results means nobody has to redo these steps by hand.

What the code does: saves the four scaled tables to `data/processed/` as `train_scaled.csv`, `val_scaled.csv`, `cal_scaled.csv` and `dataset2_scaled.csv`, and saves the fitted scaler as `scaler.joblib`. It then loads `train_scaled.csv` back (`test_load`), compares its shape with `train_scaled`, and lists the files in the folder.

In [7]:
import os
import joblib

out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

train_scaled.to_csv(f"{out_dir}/train_scaled.csv", index=False)
val_scaled.to_csv(f"{out_dir}/val_scaled.csv", index=False)
cal_scaled.to_csv(f"{out_dir}/cal_scaled.csv", index=False)
dfT2_scaled.to_csv(f"{out_dir}/dataset2_scaled.csv", index=False)

joblib.dump(scaler, f"{out_dir}/scaler.joblib")

test_load = pd.read_csv(f"{out_dir}/train_scaled.csv", parse_dates=["DATETIME"])
print(test_load.shape, train_scaled.shape)
print(os.listdir(out_dir))

(5256, 38) (5256, 38)
['cal_scaled.csv', 'dataset2_scaled.csv', 'scaler.joblib', 'train_scaled.csv', 'val_scaled.csv']


**What this shows:** the reloaded file has the same shape as `train_scaled`: 5,256 rows and 38 columns (`DATETIME`, 36 features and `ATT_FLAG`). All five files are in `data/processed/`.

Caution:
- The CSV files store `DATETIME` as text. Load them with `pd.read_csv(..., parse_dates=["DATETIME"])` to get real dates back.
- The other four tables are already scaled. `scaler.joblib` is only for scaling the Test set at the very end, after all choices and thresholds are frozen. The Test set must be rounded to 2 decimals and have the same 7 columns dropped first, because the scaler expects the same 28 continuous columns.

## End of notebook

The full write-up is in `docs/pipeline.md`. The code cell below is empty.

Decided:
- Round all data to 2 decimals (step 0b).
- Drop the 7 columns that never change (step 0b).

Still open:
- Dataset 2 labels (step 6): not done yet.
- Range: our files use 0 to 1, but the Transformer training script scales to −1 to 1.
- One shared source of data: the data loader in the repo should use these files, or at least the same rounding, drop, split and scaling.